In [ ]:
import pandas as pd

file_path = '../../../data/raw/csv/consumption_v1.csv'
df = pd.read_csv(file_path)

member_1_df = df[df['멤버 id'] == 1]


In [13]:
import os
print(os.getcwd()) 

c:\Users\user\dev\catcher-llm\notebook\team02\data_pre


In [24]:
import pandas as pd

# 1. 데이터 로드 (앞서 설정한 경로 사용)
file_path = '../../../data/raw/csv/consumption_v1.csv'
df = pd.read_csv(file_path)

member_1_df = df[df['멤버 id'] == 1]


# 2. IQR 배수를 높여서 이상치 추출 (배수를 20로 설정)
def get_extreme_outliers(df, column, multiplier=25.0):
    Q1 = df[column].quantile(0.20)
    Q3 = df[column].quantile(0.80)
    IQR = Q3 - Q1
    
    # 배수를 높여서 매우 엄격한 경계 설정
    upper_bound = Q3 + multiplier * IQR
    
    outliers = df[df[column] > upper_bound]
    
    print(f"--- 멤버 1 '{column}' 극단적 이상치 분석 (배수: {multiplier}) ---")
    print(f"Upper Bound: {upper_bound:,.0f}원")
    print(f"검출된 이상치 개수: {len(outliers)}건")
    
    return outliers


# 3. 배수를 조정하며 2건이 나오는지 확인
# 만약 2건이 안 나온다면 20, 30 등으로 숫자를 바꿔보세요.
outliers_2_items = get_extreme_outliers(df_m1, '사용 금액', multiplier=25.0)

print("\n[추출된 2개의 이상치 내역]")
print(outliers_2_items[['사용 시간', '결제 내역', '사용 금액', '업종 카테고리']])

--- 멤버 1 '사용 금액' 극단적 이상치 분석 (배수: 25.0) ---
Upper Bound: 339,688원
검출된 이상치 개수: 2건

[추출된 2개의 이상치 내역]
                사용 시간   결제 내역    사용 금액 업종 카테고리
80   2024-01-15 14:30  OO종합병원  1500000      의료
237  2024-02-15 11:00   애플스토어  2850000      쇼핑


In [28]:
import pandas as pd

file_path = '../../../data/raw/csv/consumption_v1.csv'
df = pd.read_csv(file_path)

# 멤버 1 필터링
df_m1 = df[df['멤버 id'] == 1].copy()

# 'amount' 대신 '사용 금액' 사용
def get_outliers(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = df[(df[column] < lower_bound) | (df[column] > upper_bound)]
    return outliers

# 호출 시 정확한 컬럼명 입력
outliers_df = get_outliers(df_m1, '사용 금액') # <-- 여기를 확인하세요!
print(outliers_df.head())


    멤버 id     id  사용 금액             사용 시간   결제 내역 결제 장소 (가맹점 여부) 할부 여부  할부 개월  \
0       1  30001  65000  2024-01-01 10:00  SKT통신비              Y     N      0   
6       1  30007  30947  2024-01-01 18:38   배달의민족              Y     N      0   
12      1  30013  29572  2024-01-02 18:19   배달의민족              Y     N      0   
13      1  30014  37078  2024-01-03 10:53      쿠팡              Y     N      0   
18      1  30019  28918  2024-01-03 20:14    쿠팡이츠              Y     N      0   

   할부 무/유이자 여부 거래 상태 (승인 / 취소) 해외 결제 업종 카테고리 결제 방식 (온/오프라인)  
0            -              승인     N      생활           자동이체  
6            -              승인     N      식비    온라인 - 카카오페이  
12           -              승인     N      식비    온라인 - 카카오페이  
13           -              승인     N      쇼핑    오프라인 - 삼성페이  
18           -              승인     N      식비    온라인 - 카카오페이  


In [29]:
# 1. 클리핑 처리 (상하한선 적용)
Q1 = df_m1['사용 금액'].quantile(0.25)
Q3 = df_m1['사용 금액'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

df_m1['사용 금액_clipped'] = df_m1['사용 금액'].clip(lower=lower_bound, upper=upper_bound)

# 2. 전/후 지표 비교
comparison = pd.DataFrame({
    '구분': ['원본 (Original)', '클리핑 후 (Clipped)'],
    '평균 (Mean)': [df_m1['사용 금액'].mean(), df_m1['사용 금액_clipped'].mean()],
    '중앙값 (Median)': [df_m1['사용 금액'].median(), df_m1['사용 금액_clipped'].median()],
    '표준편차 (Std)': [df_m1['사용 금액'].std(), df_m1['사용 금액_clipped'].std()],
    '최댓값 (Max)': [df_m1['사용 금액'].max(), df_m1['사용 금액_clipped'].max()]
})

print("--- 이상치 처리 전/후 통계 비교 ---")
print(comparison.round(2))

# 3. 평균 변화율 확인 (%)
mean_diff = ((df_m1['사용 금액_clipped'].mean() - df_m1['사용 금액'].mean()) / df_m1['사용 금액'].mean()) * 100
print(f"\n평균 변화율: {mean_diff:.2f}%")

--- 이상치 처리 전/후 통계 비교 ---
                구분  평균 (Mean)  중앙값 (Median)  표준편차 (Std)  최댓값 (Max)
0    원본 (Original)   20435.65        7376.0   148852.96  2850000.0
1  클리핑 후 (Clipped)    9983.40        7376.0     6875.05    22754.5

평균 변화율: -51.15%
